# Step 3: Player-role clustering (milestone 1)

Groups the 2017/18 Wyscout players into **roles within each broad position** (`Plan.md` §5, decision D6).
Code: `src/features/player_roles.py`, whose docstring documents every choice. Input:
`data/processed/player_features.csv` (run `python -m src.pipelines.build_player_features` first).

- **Players:** ≥ 450 league minutes, the same threshold as the transfer dataset.
- **Positions:** defenders, midfielders and forwards are clustered separately; goalkeepers form one role.
- **Features (12):** describe *how* someone plays, not how well (§1 of the module docstring).
  `log1p` on per-90 counts, then z-scored within each position.
- **Method:** k-means (seed 42). k is chosen per position from silhouette score, bootstrap stability
  (adjusted Rand index; 1 = identical groupings) **and** whether the clusters can be named.
- **Naming:** each role is named by an **anchor player** ("the cluster containing Bonucci"), because
  k-means numbers clusters arbitrarily. The code raises an error if anchors collide or a cluster is unnamed.

Time window: 2017/18 only, which ended before every milestone 1 transfer, so roles are pre-transfer.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"No data/raw folder found in {Path.cwd()} or its parents")
sys.path.insert(0, str(ROOT))
from src.features import player_roles as pr

OUT = ROOT / "data/processed"
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

pf = pd.read_csv(OUT / "player_features.csv")
transfers = pd.read_csv(OUT / "transfers_joined.csv")
players = pr.eligible_players(pf)
print(f"{len(pf):,} players in player_features.csv; {len(players):,} with >= {pr.MIN_MINUTES} minutes")
print(players.wy_role.value_counts().to_string())

K_BY_POSITION = {"DF": 4, "MD": 4, "FW": 3}   # chosen in section 1
ANCHORS = {                                    # role name -> anchor player's wy_id
    "DF": {"Attacking full-back": 7885,        # R. Bertrand
           "Defensive centre-back": 8653,      # H. Maguire
           "Ball-playing centre-back": 20459,  # L. Bonucci
           "Defensive full-back": 8313},       # P. Zabaleta
    "MD": {"Ball-winning midfielder": 4488,    # Saúl
           "Wide midfielder": 14812,           # I. Perišić
           "Deep-lying midfielder": 3346,      # Sergio Busquets
           "Attacking midfielder": 38021},     # K. De Bruyne
    "FW": {"Creative forward": 3359,           # L. Messi
           "Wide support forward": 5400,       # Gerard Moreno
           "Central striker": 8717},           # H. Kane
}

2,561 players in player_features.csv; 1,947 with >= 450 minutes
wy_role
MD    706
DF    703
FW    403
GK    135


## 1. Choosing k per position
Silhouette always prefers k = 2, which here only separates central from wide players. Stability is
what separates the useful options: a k whose clusters reappear on resampled data (ARI close to 1)
describes real structure, not chance. *Runs 50 bootstrap refits per k (~2 minutes).*

In [2]:
for pos in pr.OUTFIELD:
    g = players[players.wy_role == pos].reset_index(drop=True)
    print(f"=== {pos} ({len(g)} players)")
    display(pr.evaluate_k(pr.position_matrix(g)).round(3))

=== DF (703 players)


,k,silhouette,stability_ari,stability_sd,smallest_cluster,largest_cluster
0,2,0.366,0.993,0.011,327,376
1,3,0.243,0.755,0.255,148,319
2,4,0.188,0.891,0.046,151,217
3,5,0.181,0.801,0.083,70,216
4,6,0.154,0.692,0.111,95,149


=== MD (706 players)


,k,silhouette,stability_ari,stability_sd,smallest_cluster,largest_cluster
0,2,0.322,0.966,0.019,311,395
1,3,0.194,0.654,0.218,203,285
2,4,0.169,0.786,0.133,153,189
3,5,0.156,0.641,0.119,111,171
4,6,0.151,0.655,0.112,91,149


=== FW (403 players)


,k,silhouette,stability_ari,stability_sd,smallest_cluster,largest_cluster
0,2,0.236,0.904,0.047,176,227
1,3,0.218,0.884,0.060,100,195
2,4,0.152,0.616,0.149,79,140
3,5,0.156,0.632,0.157,61,131
4,6,0.144,0.559,0.117,48,112


**Choice:**
- **Defenders: k = 4** (ARI 0.89). k = 3 is unstable (0.76 ± 0.26), and k = 2 is just centre-backs vs full-backs.
- **Midfielders: k = 4** (0.79). k = 3, 5 and 6 are less stable, and k = 2 is central vs wide.
- **Forwards: k = 3** (0.88, almost as stable as k = 2). k ≥ 4 drops to ≤ 0.63.

All four roles per position can be named from their profiles (section 2).

## 2. Role profiles
Values are mean z-scores **within the position**: +1 means one standard deviation above that position's
average. Examples are the players with the most minutes in each role.

In [3]:
roles = pr.assign_roles(pf, K_BY_POSITION, ANCHORS)
short = {"passes_p90": "pass", "pass_acc": "acc", "progressive_passes_p90": "prog", "key_passes_p90": "key",
         "crosses_p90": "cross", "shots_p90": "shot", "dribbles_p90": "drib", "duels_p90": "duel",
         "interceptions_p90": "intc", "avg_x": "x", "avg_width": "width", "final_third_share": "f3rd"}
for pos in pr.OUTFIELD:
    g = players[players.wy_role == pos].reset_index(drop=True)
    r = roles[roles.position == pos].set_index("wy_id").loc[g.wy_id]
    prof = pr.cluster_profiles(g, r.role_cluster.to_numpy(), pr.position_matrix(g)).rename(columns=short)
    prof.index = [r.drop_duplicates("role_cluster").set_index("role_cluster").role[c] for c in prof.index]
    g2 = g.assign(role=r.role.to_numpy())
    prof["examples (most minutes)"] = [", ".join(g2[g2.role == n].nlargest(6, "wy_minutes").shortName) for n in prof.index]
    print(f"=== {pos}"); display(prof.round(1))

=== DF


,n,pass,acc,prog,key,cross,shot,drib,duel,intc,x,width,f3rd,examples (most minutes)
Attacking full-back,154,0.4,-0.4,0.5,1.3,1.3,0.9,1.0,0.5,-0.8,1.3,1.0,1.3,"R. Bertrand, E. Hysaj, R. Rosales, Y. Sabaly, ..."
Defensive centre-back,217,-0.6,0.3,-0.7,-0.7,-0.9,-0.3,-0.7,-0.4,0.6,-1.0,-0.9,-0.9,"H. Maguire, L. Dunk, F. Acerbi, M. Jørgensen, ..."
Ball-playing centre-back,151,1.1,1.0,0.8,-0.6,-0.8,0.0,-0.6,-0.4,0.0,-0.6,-0.8,-0.9,"M. Škriniar, David López, Azpilicueta, K. Glik..."
Defensive full-back,181,-0.6,-0.8,-0.3,0.2,0.7,-0.4,0.5,0.4,-0.1,0.6,0.9,0.6,"P. Zabaleta, F. Guilbert, Jonny, K. Amian, M. ..."


=== MD


,n,pass,acc,prog,key,cross,shot,drib,duel,intc,x,width,f3rd,examples (most minutes)
Ball-winning midfielder,189,-0.4,0.1,-0.6,-0.7,-0.6,-0.4,-0.3,0.2,0.5,-0.6,-0.4,-0.6,"J. Cork, R. Cohade, L. Lerager, R. Mandragora,..."
Wide midfielder,182,-0.9,-1.0,-0.6,0.5,1.0,0.6,1.0,0.6,-0.8,1.1,1.0,1.2,"I. Perišić, Rômulo, A. Townsend, M. Lazzari, G..."
Deep-lying midfielder,182,0.9,0.9,0.7,-0.6,-0.9,-0.7,-0.8,-0.7,0.7,-1.0,-0.9,-1.0,"A. Doucouré, G. Xhaka, D. Stephens, Rodri, L. ..."
Attacking midfielder,153,0.5,0.1,0.7,0.9,0.6,0.6,0.2,-0.2,-0.5,0.6,0.4,0.5,"F. Kessié, C. Eriksen, J. Veretout, S. Padoin,..."


=== FW


,n,pass,acc,prog,key,cross,shot,drib,duel,intc,x,width,f3rd,examples (most minutes)
Creative forward,100,1.0,0.4,1.0,1.1,0.9,0.5,0.7,-0.4,-0.3,0.5,0.5,0.6,"José Callejón, L. Insigne, K. Toko Ekambi, D. ..."
Wide support forward,108,0.4,-0.1,0.5,-0.1,0.4,-0.6,0.3,0.0,1.0,-1.1,0.8,-1.0,"Gerard Moreno, J. Briand, R. Rodelin, J. Ayew,..."
Central striker,195,-0.7,-0.2,-0.8,-0.6,-0.7,0.1,-0.6,0.2,-0.4,0.4,-0.7,0.2,"J. Vardy, J. Calleri, H. Kane, E. Sala, Maxi G..."


## 3. Famous-player check
Do well-known players land where a scout would put them?

In [4]:
check = ["Marcelo", "J. Kimmich", "Sergio Ramos", "M. Hummels", "Sergio Busquets", "Jorginho", "N. Kanté",
         "Casemiro", "L. Modrić", "K. De Bruyne", "L. Messi", "Neymar", "A. Griezmann", "H. Kane", "R. Lewandowski"]
fam = roles.merge(pf[["wy_id", "shortName", "wy_team", "wy_minutes"]], on="wy_id")
display(fam[fam.shortName.isin(check)].sort_values(["position", "role"])[["shortName", "wy_team", "position", "role"]])

,shortName,wy_team,position,role
32,Marcelo,Real Madrid,DF,Attacking full-back
566,J. Kimmich,Bayern München,DF,Attacking full-back
4,Marcelo,Olympique Lyonnais,DF,Ball-playing centre-back
29,Sergio Ramos,Real Madrid,DF,Ball-playing centre-back
204,M. Hummels,Bayern München,DF,Ball-playing centre-back
777,H. Kane,Tottenham Hotspur,FW,Central striker
804,R. Lewandowski,Bayern München,FW,Central striker
722,L. Messi,Barcelona,FW,Creative forward
733,A. Griezmann,Atlético Madrid,FW,Creative forward
936,Neymar,PSG,FW,Creative forward


**Reading:** full-backs, centre-backs, playmakers and forwards land where expected.

**One revealing case:** Kanté and Casemiro, classic ball-winners, share the *deep-lying midfielder* role
with Busquets and Jorginho, while the *ball-winning* role is mostly midfielders from teams with less
possession (Cork, Allen).
- **Why:** per-90 counts depend partly on **how much of the ball the player's team has**. A holding
  midfielder at Chelsea passes far more than one at Burnley.
- **What we did:** named the role "deep-lying midfielder", not "deep-lying playmaker".
- **Possible improvement:** possession-adjusted stats (e.g. per 1,000 team touches), part of the
  team-style work.

## 4. Roles of the transferred players (descriptive only)
Mean playing-time share after the transfer, by the player's role before it. With about 20–40 transfers
per role, these differences are noisy. Whether roles help predict y is decided later, by
cross-validation on the training rows (`Plan.md` §4).

In [5]:
t = transfers[["wy_id", "y"]].merge(roles[["wy_id", "position", "role"]], on="wy_id", how="left")
print(f"{t.role.notna().sum()} of {len(t)} transfers have a role")
display(t.groupby(["position", "role"]).y.agg(["count", "mean", "std"]).round(3))

276 of 276 transfers have a role


count   mean    std
position role                                         
DF       Attacking full-back          19  0.435  0.282
         Ball-playing centre-back     19  0.538  0.307
         Defensive centre-back        19  0.526  0.329
         Defensive full-back          23  0.431  0.316
FW       Central striker              30  0.561  0.242
         Creative forward             17  0.491  0.307
         Wide support forward         20  0.400  0.237
GK       Goalkeeper                   23  0.559  0.398
MD       Attacking midfielder         20  0.366  0.295
         Ball-winning midfielder      36  0.504  0.240
         Deep-lying midfielder        24  0.554  0.236
         Wide midfielder              26  0.457  0.294

In [6]:
roles.to_csv(OUT / "player_roles.csv", index=False)
print(f"Saved {len(roles):,} players to {OUT / 'player_roles.csv'}")
display(roles.groupby(["position", "role"]).size().rename("players"))

Saved 1,947 players to /home/slage/Coding/Deep Learning/Project/Deep-Learning-Scouting/data/processed/player_roles.csv


position  role                    
DF        Attacking full-back         154
          Ball-playing centre-back    151
          Defensive centre-back       217
          Defensive full-back         181
FW        Central striker             195
          Creative forward            100
          Wide support forward        108
GK        Goalkeeper                  135
MD        Attacking midfielder        153
          Ball-winning midfielder     189
          Deep-lying midfielder       182
          Wide midfielder             182
Name: players, dtype: int64